# NB07: Corrective RAG (CRAG) for Biomedical Questions

## What
Corrective RAG (CRAG) is a quality-controlled retrieval-generation workflow that explicitly grades retrieval quality and applies corrective actions when retrieval is weak.

## Why
Biomedical QA is high-risk for hallucination when retrieval quality is low. CRAG reduces this risk by introducing:
- explicit retrieval grading,
- query correction/retrieval retry,
- fallback evidence acquisition,
- post-generation grounding verification.

## When
Use CRAG when:
- retrieval quality is inconsistent across question types,
- domain coverage is sparse or long-tail,
- groundedness requirements are strict (medical safety).

## Tradeoffs
- Better reliability than one-pass RAG.
- Increased complexity and latency due to corrective loops.
- Requires threshold tuning and route observability.

## Alternatives
- **Single-pass RAG**: lower latency, less robust when retrieval fails.
- **Agentic RAG without explicit corrective policy**: flexible, but can be harder to audit against deterministic correction criteria.
- **Rerank-only pipelines**: helpful but may still fail when initial recall is weak.

## Production Considerations
- Track route frequencies (`accept`, `correct`, `web_fallback`) as health signals.
- Cap correction attempts to control cost.
- Use strict model output schemas for graders.
- Log complete state traces for replay and incident analysis.

## Definition and Core Concepts
- **CRAG (Corrective RAG)**: retrieval-quality-aware RAG with explicit corrective branches.
- **Retrieval Grader**: a quality gate deciding whether to accept, correct, or fallback.
- **Corrective Loop**: bounded retries that improve query/context before generation.
- **Verification Gate**: post-answer groundedness check with optional corrective fallback.

CRAG is a reliability overlay, not a replacement for baseline retrieval components.

## Why CRAG Was Developed
One-pass RAG pipelines often fail silently when retrieval quality is weak.
CRAG introduces deterministic control points so failures become observable and recoverable.

## What Traditional RAG Limitation It Solves
Traditional RAG assumes retrieved context is sufficient. CRAG addresses:
- low-quality retrieval acceptance,
- lack of corrective query reformulation,
- weak fallback behavior under sparse evidence,
- insufficient post-answer grounding verification.

## Architecture Explanation
- `Hybrid Retriever`: initial candidate retrieval.
- `Judge Retrieval Grader`: quality score + missing-aspect hints.
- `Query Correction`: reformulate query using grader feedback.
- `Web Fallback`: bounded external evidence path for unresolved gaps.
- `Answer + Verify`: generate then verify groundedness before final response.

## Workflow Explanation
1. Retrieve with hybrid search.
2. Grade retrieval quality.
3. Branch:
   - accept context,
   - retry with corrected query,
   - fallback to web evidence.
4. Generate answer and verify grounding.
5. Finalize or perform one bounded corrective verify loop.

## Component-by-Component Breakdown
1. **State Object (`CRAGState`)**: carries route metadata and trace.
2. **Routing Nodes**: retrieval, grading, correction, fallback, generation, verification.
3. **Safety Controls**: max correction count and verify-attempt bounds.
4. **Observability**: route traces and route summary tables.

## Advantages and Disadvantages
**Advantages**
- More robust under weak retrieval.
- Transparent route-level diagnostics.
- Better guardrails for biomedical grounding.

**Disadvantages**
- Higher latency/cost from extra model calls.
- More threshold tuning complexity.
- Requires disciplined trace logging and monitoring.

## Comparison Against Other Implemented Variants
- **Standard RAG**: fastest path, weakest correction behavior.
- **Hybrid RAG**: improves retrieval quality, but no explicit corrective controller.
- **Agentic GraphRAG**: broader orchestration capabilities; CRAG is tighter reliability policy.
- **CRAG (this notebook)**: explicit quality-driven correction and bounded fallback.

## Implementation Decisions in This Project
- Retrieval uses existing hybrid retriever from NB06.
- Grading and groundedness use `granite4.1:8b`.
- Query correction uses generator model with deterministic low-temperature rewrite prompt.
- Web fallback is bounded and used only after explicit routing criteria.

## CRAG Architecture Diagram

```mermaid
flowchart TD
    Q[Biomedical Query] --> R[Hybrid Retriever]
    R --> G[Judge Retrieval Grader]
    G -->|High quality| C[Context Build]
    G -->|Low quality + retries left| QC[Query Correction]
    QC --> R
    G -->|Low quality + retries exhausted| WF[Web Fallback]
    WF --> C
    C --> A[Answer Generation<br/>granite4.1:8b]
    A --> V[Judge Verification<br/>granite4.1:8b]
    V -->|Grounded| F[Final Response]
    V -->|Not grounded (bounded retry)| WF
```

## Workflow Rationale

CRAG differs from basic RAG because retrieval quality is a first-class signal controlling downstream behavior.
This notebook implements CRAG as a standalone additive state machine without modifying existing GraphRAG/Agentic pipelines.

In [1]:
# Input: existing corpus artifacts and additive CRAG/hybrid modules.
# Output: initialized runtime for standalone CRAG pipeline.
# Logic: load chunks, sparse index, and chroma collection, then build CRAG graph.
# Complexity: O(number_of_chunks + index_build).
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.chroma_retriever import get_collection
from src.chunking import build_chunk_lookup, load_chunks
from src.config import settings
from src.crag_pipeline import CRAGResources, build_crag_workflow, crag_mermaid, run_crag_batch
from src.data_pipeline import build_extractive_eval_queries, load_persisted_records
from src.evaluator import (
    GenerationExample,
    RetrievalExample,
    compute_generation_metrics,
    compute_rag_metrics,
    compute_retrieval_metrics,
)
from src.hybrid_retriever import BiomedicalSparseIndex
from src.utils import save_json

RUN_FULL_EVAL = os.getenv("RUN_FULL_EVAL", "false").strip().lower() == "true"
print(f"CRAG acceptance threshold: {settings.crag_acceptance_threshold}")
print(f"CRAG max corrections: {settings.crag_max_corrections}")
print(f"Judge model: {settings.judge_model}")
print(f"RUN_FULL_EVAL={RUN_FULL_EVAL}")

CRAG acceptance threshold: 0.55
CRAG max corrections: 2
Judge model: granite4.1:8b
RUN_FULL_EVAL=True


## Step 1: Load Corpus + Build Sparse Index

In [2]:
# Input: persisted MedMentions records/chunks.
# Output: eval query set and sparse index for hybrid retrieval.
# Logic: hydrate data and build lexical retriever needed by CRAG retrieval node.
# Complexity: O(total_tokens).
records = load_persisted_records()
chunks = load_chunks()
chunk_lookup = build_chunk_lookup(chunks)

try:
    collection = get_collection("medmentions_chroma_section_a")
except Exception:
    collection = get_collection("medmentions_chroma")

sparse_index = BiomedicalSparseIndex()
sparse_index.fit(chunks)

eval_queries = build_extractive_eval_queries(
    records=records,
    chunk_lookup=chunk_lookup,
    sample_size=settings.eval_query_count,
)

print(f"Records: {len(records):,}")
print(f"Chunks: {len(chunks):,}")
print(f"Eval queries: {len(eval_queries):,}")

2026-06-22 11:08:44 | INFO | Built sparse biomedical index over 5098 documents
2026-06-22 11:08:44 | INFO | fit completed in 0.32s
2026-06-22 11:08:45 | INFO | Built 8 extractive evaluation queries
Records: 4,392
Chunks: 5,098
Eval queries: 8


## Step 2: Build Standalone CRAG State Machine

We keep this CRAG pipeline separate from existing agentic code to remain strictly additive.

In [3]:
# Input: chroma collection and sparse index resources.
# Output: compiled CRAG app.
# Logic: instantiate CRAG resource bundle and compile LangGraph workflow.
# Complexity: O(1) graph construction.
resources = CRAGResources(
    chroma_collection=collection,
    sparse_index=sparse_index,
)
app = build_crag_workflow(resources)
print("CRAG workflow compiled.")
print("\nCRAG Mermaid:\n")
print(crag_mermaid())

CRAG workflow compiled.

CRAG Mermaid:

flowchart TD
    A[Query] --> B[Hybrid Retrieval]
    B --> C[Retrieval Grader]
    C -->|High quality| D[Context Expansion]
    C -->|Low quality, retries left| E[Query Correction]
    E --> B
    C -->|Low quality, retries exhausted| F[Web Fallback]
    F --> D
    D --> G[Answer Generation]
    G --> H[Judge Verification]
    H -->|Grounded| I[Final Response]
    H -->|Ungrounded, one retry| F


## Step 3: Qualitative CRAG Route Demonstration

We inspect route decisions and traces for representative biomedical queries.

In [4]:
# Input: demonstration biomedical queries.
# Output: CRAG final states including route traces.
# Logic: run batch CRAG inference and inspect corrective behaviors.
# Complexity: O(num_queries * CRAG_cost).
demo_queries = [
    "What evidence links diabetes with insulin resistance in this corpus?",
    "Summarize findings about hypertension risk factors.",
    "What do retrieved abstracts report about KRAS in pancreatic cancer?",
]

demo_states = run_crag_batch(app, demo_queries)

for idx, state in enumerate(demo_states, start=1):
    print(f"\n==== CRAG Demo {idx} ====")
    print("Query:", state.get("query", ""))
    print("Corrected query:", state.get("corrected_query", ""))
    print("Retrieval grade:", state.get("retrieval_grade", 0.0))
    print("Groundedness:", state.get("groundedness", 0.0))
    print("Route:", state.get("route", ""))
    print("Trace:", " -> ".join(state.get("trace", [])))
    print("Answer preview:", state.get("final_answer", "")[:420], "...")

2026-06-22 11:08:50 | INFO | Embedded batch 1/1 (1 texts)
2026-06-22 11:08:50 | INFO | Embedded 1 texts with model qwen3-embedding:4b
2026-06-22 11:08:50 | INFO | embed_texts completed in 5.37s


2026-06-22 11:09:43 | INFO | Embedded batch 1/1 (1 texts)
2026-06-22 11:09:43 | INFO | Embedded 1 texts with model qwen3-embedding:4b
2026-06-22 11:09:43 | INFO | embed_texts completed in 5.48s


2026-06-22 11:10:42 | INFO | Embedded batch 1/1 (1 texts)
2026-06-22 11:10:42 | INFO | Embedded 1 texts with model qwen3-embedding:4b
2026-06-22 11:10:42 | INFO | embed_texts completed in 5.77s


2026-06-22 11:11:04 | INFO | Embedded batch 1/1 (1 texts)
2026-06-22 11:11:04 | INFO | Embedded 1 texts with model qwen3-embedding:4b
2026-06-22 11:11:04 | INFO | embed_texts completed in 5.59s


2026-06-22 11:11:33 | INFO | Embedded batch 1/1 (1 texts)
2026-06-22 11:11:33 | INFO | Embedded 1 texts with model qwen3-embedding:4b
2026-06-22 11:11:33 | INFO | embed_texts completed in 5.57s


/home/ahmad/AI/Medical-Research-GraphRAG/src/crag_pipeline.py:58: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS(timeout=12) as ddgs:


/home/ahmad/AI/Medical-Research-GraphRAG/src/crag_pipeline.py:58: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS(timeout=12) as ddgs:



==== CRAG Demo 1 ====
Query: What evidence links diabetes with insulin resistance in this corpus?
Corrected query: What evidence links diabetes with insulin resistance in this corpus?
Retrieval grade: 0.7
Groundedness: 0.9
Route: finalize
Trace: retrieve_hybrid -> grade_retrieval -> context_expansion -> answer_generation -> verify_answer -> finalize
Answer preview: The corpus provides several pieces of evidence that link diabetes with insulin resistance:

1. **Evidence from Source [1]**: This source discusses the relationship between rheumatoid arthritis (RA), insulin resistance, and diabetes. It states that "the increased insulin resistance seen in RA is closely linked to the systemic inflammation induced by certain proinflammatory cytokines such as tumor necrosis factor α (TN ...

==== CRAG Demo 2 ====
Query: Summarize findings about hypertension risk factors.
Corrected query: Summarize findings about hypertension risk factors.
Retrieval grade: 0.62
Groundedness: 0.78
Route: finaliz

## Step 4: Route Analytics

We summarize corrective behavior frequencies to understand reliability and cost tradeoffs.

In [5]:
# Input: CRAG run states.
# Output: route analytics table.
# Logic: aggregate route outcomes and corrective attempt counts.
# Complexity: O(num_states).
route_df = pd.DataFrame(
    [
        {
            "query": row.get("query", ""),
            "corrected_query": row.get("corrected_query", ""),
            "retrieval_grade": float(row.get("retrieval_grade", 0.0)),
            "groundedness": float(row.get("groundedness", 0.0)),
            "hallucination_risk": float(row.get("hallucination_risk", 0.0)),
            "completeness": float(row.get("completeness", 0.0)),
            "correction_attempts": int(row.get("correction_attempts", 0)),
            "verify_attempts": int(row.get("verify_attempts", 0)),
            "route": row.get("route", ""),
            "trace": " -> ".join(row.get("trace", [])),
        }
        for row in demo_states
    ]
)
route_df

,query,corrected_query,retrieval_grade,groundedness,hallucination_risk,completeness,correction_attempts,verify_attempts,route,trace
0,What evidence links diabetes with insulin resi...,What evidence links diabetes with insulin resi...,0.70,0.90,0.10,0.80,0,0,finalize,retrieve_hybrid -> grade_retrieval -> context_...
1,Summarize findings about hypertension risk fac...,Summarize findings about hypertension risk fac...,0.62,0.78,0.12,0.68,0,0,finalize,retrieve_hybrid -> grade_retrieval -> context_...
2,What do retrieved abstracts report about KRAS ...,What do retrieved abstracts report on the prev...,0.20,0.20,0.90,0.15,2,1,finalize,retrieve_hybrid -> grade_retrieval -> query_co...


## Step 5: CRAG Evaluation Harness

Required metric families are prepared here:
- Retrieval metrics: Precision@K, Recall@K, F1, MRR, NDCG
- Generation metrics: Exact Match, BLEU, ROUGE, METEOR, BERTScore
- RAG metrics: Faithfulness, Context Precision, Context Recall, Answer Relevancy
- LLM-as-Judge with `granite4.1:8b`

In [6]:
# Input: eval query set and CRAG app.
# Output: retrieval/generation/rag metric payloads.
# Logic: run CRAG on eval queries, derive retrieval/generation examples, and score metrics.
# Complexity: O(num_queries * CRAG_workflow_cost).
retrieval_examples: list[RetrievalExample] = []
generation_examples: list[GenerationExample] = []

if RUN_FULL_EVAL:
    states = run_crag_batch(app, [item.query for item in eval_queries])
    for item, state in zip(eval_queries, states):
        retrieved_ids = [row["id"] for row in state.get("retrieval_rows", [])]
        retrieval_examples.append(
            RetrievalExample(retrieved_ids=retrieved_ids, relevant_ids=item.supporting_chunk_ids)
        )

        generation_examples.append(
            GenerationExample(
                query=item.query,
                answer=state.get("final_answer", ""),
                reference_answer=item.reference_answer,
                context_chunks=[row.get("text", "") for row in state.get("retrieval_rows", [])[:8]],
            )
        )

    retrieval_metrics = compute_retrieval_metrics(retrieval_examples, k_values=[1, 3, 5, 8])
    generation_metrics = compute_generation_metrics(generation_examples, include_bertscore=True)
    rag_metrics = compute_rag_metrics(generation_examples)
else:
    retrieval_metrics = {
        "precision@1": None,
        "precision@3": None,
        "precision@5": None,
        "precision@8": None,
        "recall@1": None,
        "recall@3": None,
        "recall@5": None,
        "recall@8": None,
        "f1@1": None,
        "f1@3": None,
        "f1@5": None,
        "f1@8": None,
        "ndcg@1": None,
        "ndcg@3": None,
        "ndcg@5": None,
        "ndcg@8": None,
        "mrr": None,
        "placeholder_note": "Populate by setting RUN_FULL_EVAL=True in explicit execution phase.",
    }
    generation_metrics = {
        "exact_match": None,
        "bleu": None,
        "rouge1": None,
        "rouge2": None,
        "rougeL": None,
        "meteor": None,
        "bertscore_precision": None,
        "bertscore_recall": None,
        "bertscore_f1": None,
        "placeholder_note": "Populate by setting RUN_FULL_EVAL=True in explicit execution phase.",
    }
    rag_metrics = {
        "faithfulness": None,
        "context_precision": None,
        "context_recall": None,
        "answer_relevancy": None,
        "judge_groundedness": None,
        "judge_relevance": None,
        "judge_hallucination": None,
        "judge_completeness": None,
        "placeholder_note": "Populate by setting RUN_FULL_EVAL=True in explicit execution phase.",
    }

pd.DataFrame([retrieval_metrics])
pd.DataFrame([generation_metrics])
pd.DataFrame([rag_metrics])

/home/ahmad/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/collections/__init__.py:449: ResourceWarning: unclosed <socket.socket fd=71, family=2, type=1, proto=6, laddr=('127.0.0.1', 58124), raddr=('127.0.0.1', 11434)>
  result = tuple_new(cls, iterable)
/home/ahmad/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/collections/__init__.py:449: ResourceWarning: unclosed <socket.socket fd=83, family=2, type=1, proto=6, laddr=('127.0.0.1', 36588), raddr=('127.0.0.1', 11434)>
  result = tuple_new(cls, iterable)
/home/ahmad/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/collections/__init__.py:449: ResourceWarning: unclosed <socket.socket fd=84, family=2, type=1, proto=6, laddr=('127.0.0.1', 35542), raddr=('127.0.0.1', 11434)>
  result = tuple_new(cls, iterable)
/home/ahmad/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/collections/__init__.py:449: ResourceWarning: unclosed <socket.socket fd=85, fam

2026-06-22 11:13:25 | INFO | Embedded batch 1/1 (1 texts)
2026-06-22 11:13:25 | INFO | Embedded 1 texts with model qwen3-embedding:4b
2026-06-22 11:13:25 | INFO | embed_texts completed in 5.42s


/home/ahmad/AI/Medical-Research-GraphRAG/src/hybrid_retriever.py:149: ResourceWarning: unclosed <socket.socket fd=71, family=2, type=1, proto=6, laddr=('127.0.0.1', 35632), raddr=('127.0.0.1', 11434)>
  ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]


2026-06-22 11:14:09 | INFO | Embedded batch 1/1 (1 texts)
2026-06-22 11:14:09 | INFO | Embedded 1 texts with model qwen3-embedding:4b
2026-06-22 11:14:09 | INFO | embed_texts completed in 5.45s


/home/ahmad/AI/Medical-Research-GraphRAG/src/hybrid_retriever.py:149: ResourceWarning: unclosed <socket.socket fd=71, family=2, type=1, proto=6, laddr=('127.0.0.1', 35636), raddr=('127.0.0.1', 11434)>
  ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
/home/ahmad/AI/Medical-Research-GraphRAG/src/hybrid_retriever.py:149: ResourceWarning: unclosed <socket.socket fd=83, family=2, type=1, proto=6, laddr=('127.0.0.1', 58450), raddr=('127.0.0.1', 11434)>
  ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
/home/ahmad/AI/Medical-Research-GraphRAG/src/hybrid_retriever.py:149: ResourceWarning: unclosed <socket.socket fd=84, family=2, type=1, proto=6, laddr=('127.0.0.1', 56940), raddr=('127.0.0.1', 11434)>
  ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
/home/ahmad/AI/Medical-Research-GraphRAG/src/hybrid_retriever.py:149: ResourceWarning: unclosed <socket.socket fd=85, family=2, type=1, proto=6, laddr=('127.0.0.1', 5694

...

/home/ahmad/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=721251) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


/home/ahmad/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=721251) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


/home/ahmad/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=721251) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


2026-06-22 11:28:17 | INFO | compute_rag_metrics completed in 174.26s


,faithfulness,context_precision,context_recall,answer_relevancy,judge_groundedness,judge_relevance,judge_hallucination,judge_completeness
0,0.89375,0.151786,0.4,0.98125,3.875,4.625,5.0,3.375


## Step 6: Persist CRAG Artifacts

Artifacts are schema-complete now and can be populated with real values during the explicit execution phase.

In [7]:
# Input: route analytics and metric payloads.
# Output: persisted CRAG tables and JSON metrics payload.
# Logic: save stable CRAG report structure for downstream reporting.
# Complexity: O(payload_size).
route_df.to_csv(settings.tables_dir / "nb07_crag_route_summary.csv", index=False)

crag_payload = {
    "mode": "placeholder" if not RUN_FULL_EVAL else "executed",
    "workflow_mermaid": crag_mermaid(),
    "retrieval_metrics": retrieval_metrics,
    "generation_metrics": generation_metrics,
    "rag_metrics": rag_metrics,
    "route_summary_preview": route_df.head(20).to_dict(orient="records"),
    "notes": {
        "phase": "implementation_only_no_execution" if not RUN_FULL_EVAL else "executed",
        "judge_model": settings.judge_model,
    },
}
save_json(crag_payload, settings.metrics_dir / "nb07_crag_metrics.json")

print("Saved NB07 CRAG artifacts (placeholders unless RUN_FULL_EVAL=True).")

Saved NB07 CRAG artifacts (placeholders unless RUN_FULL_EVAL=True).


## Post-Run Result Analysis Template (Populate After Execution)
- Analyze route distribution (`accept`, `correct`, `web_fallback`) and what it implies about corpus coverage.
- Interpret retrieval/generation/RAG metric changes vs non-CRAG baseline.
- Interpret judge metrics (`granite4.1:8b`) for groundedness, relevance, hallucination, completeness.
- Quantify latency/cost overhead from corrective loops and fallback paths.
- Summarize where CRAG improved biomedical reliability and where it did not.